# SAR-to-EO Image Translation — Kaggle Training Notebook

**GalaxEye Technical Assignment.** This notebook runs the whole pipeline end-to-end on Kaggle's free **T4 GPU**:
data prep → train the full Pix2Pix model → train the L1-only ablation → evaluate both on the val & test splits → save loss curves and qualitative triplets.

### Before you run
1. Top-right **⋮ → Accelerator → GPU T4 x1**.
2. **+ Add Input** → search `sentinel12-image-pairs-segregated-by-terrain` (by *requiemonk*) → Add.
3. Set `REPO_URL` below to your **public GitHub repo** (the one you'll submit). The notebook clones it.
4. **Run All**. First a quick *smoke test* (few epochs, subset) verifies everything, then you scale up.


## 1. Settings

In [ ]:
# ---- EDIT THESE ----
REPO_URL   = 'https://github.com/<your-username>/sar2eo.git'  # your public repo
VAL_TERRAIN  = 'grassland'   # terrain held out for validation
TEST_TERRAIN = 'urban'       # terrain held out for test

# Smoke test first (fast, proves the pipeline). Set SMOKE=False for the real run.
SMOKE = True
SMOKE_EPOCHS = 3
SMOKE_SUBSET = 300     # pairs per terrain during smoke test
FULL_EPOCHS  = 200     # real-run epochs (matches config.yaml)
# --------------------
import os, glob, sys, subprocess

# Auto-locate the attached Kaggle dataset (the folder that holds the terrain dirs)
cands = glob.glob('/kaggle/input/*sentinel*') + glob.glob('/kaggle/input/*')
DATA_SRC = next((c for c in cands if os.path.isdir(c)), None)
print('Dataset input dir :', DATA_SRC)
assert DATA_SRC, 'No /kaggle/input dataset found — did you Add the dataset as Input?'


## 2. Get the code

In [ ]:
WORK = '/kaggle/working/sar2eo'
if REPO_URL and '<your-username>' not in REPO_URL:
    if not os.path.exists(WORK):
        subprocess.run(['git','clone','--depth','1',REPO_URL,WORK], check=True)
else:
    # Fallback: code uploaded as a Kaggle dataset/input — copy it in.
    src_code = next((c for c in glob.glob('/kaggle/input/*') if os.path.exists(os.path.join(c,'train.py'))), None)
    assert src_code, 'Set REPO_URL, or attach your code as an Input dataset containing train.py.'
    subprocess.run(['cp','-r',src_code,WORK], check=True)
os.chdir(WORK)
print('Working dir:', os.getcwd())
print(os.listdir('.'))


## 3. Dependencies
Kaggle already has PyTorch + CUDA. We only add the metric/util packages.

In [ ]:
!pip -q install lpips==0.1.4 pytorch-fid==0.3.0 torchmetrics==1.3.2 pyyaml==6.0.1 tqdm==4.66.2
import torch; print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())


## 4. Prepare data (terrain → train/val/test)

In [ ]:
subset = ['--max_per_terrain', str(SMOKE_SUBSET)] if SMOKE else []
cmd = ['python','prepare_data.py','--src',DATA_SRC,'--dst','dataset',
       '--val_terrain',VAL_TERRAIN,'--test_terrain',TEST_TERRAIN] + subset
print(' '.join(cmd))
subprocess.run(cmd, check=True)


## 5. Build the two config variants (ablation)
Identical settings except the **loss** and the output folder.

In [ ]:
import yaml, copy
base = yaml.safe_load(open('config.yaml'))
epochs = SMOKE_EPOCHS if SMOKE else FULL_EPOCHS

def write_cfg(path, mode, out_dir):
    c = copy.deepcopy(base)
    c['loss']['mode'] = mode
    c['experiment']['output_dir'] = out_dir
    c['training']['epochs'] = epochs
    if not SMOKE:
        c['training']['lr_decay_start_epoch'] = epochs // 2
    c['data']['data_root'] = 'dataset'
    c['training']['num_workers'] = 2
    yaml.safe_dump(c, open(path,'w'), sort_keys=False)
    print('wrote', path, '| mode=', mode, '| epochs=', epochs, '| out=', out_dir)

write_cfg('config_full.yaml', 'full_gan', 'outputs_full')
write_cfg('config_l1.yaml',   'l1_only',  'outputs_l1')


## 6. Train — Model B (full Pix2Pix: L1 + adversarial)

In [ ]:
!python train.py --config config_full.yaml


## 7. Train — Model A (L1-only ablation)

In [ ]:
!python train.py --config config_l1.yaml


## 8. Evaluate both models on val and test splits
Runs inference then computes LPIPS / FID / SSIM / PSNR. Results saved as JSON.

In [ ]:
import json
runs = {'full':'outputs_full', 'l1':'outputs_l1'}
results = {}
for tag, out in runs.items():
    w = f'{out}/final_weights.pth'
    for split in ['val','test']:
        pred = f'{out}/{split}_preds'
        subprocess.run(['python','infer.py','--input_dir',f'dataset/{split}/s1',
                        '--output_dir',pred,'--weights',w], check=True)
        mjson = f'{out}/metrics_{split}.json'
        subprocess.run(['python','eval.py','--pred_dir',pred,
                        '--gt_dir',f'dataset/{split}/s2','--save_json',mjson], check=True)
        results[(tag,split)] = json.load(open(mjson))

# Pretty summary table
print('\n=== RESULTS (paste into README + report) ===')
hdr = f"{'Model':<22}{'Split':<6}{'LPIPS':>9}{'FID':>9}{'SSIM':>9}{'PSNR':>9}"
print(hdr); print('-'*len(hdr))
names = {'full':'Pix2Pix (L1+GAN)', 'l1':'L1 only (ablation)'}
for tag in ['l1','full']:
    for split in ['val','test']:
        m = results[(tag,split)]
        print(f"{names[tag]:<22}{split:<6}{m['LPIPS']:>9.4f}{m['FID']:>9.2f}{m['SSIM']:>9.4f}{m['PSNR']:>9.2f}")


## 9. Collect deliverables for the ZIP
Gathers loss curves + qualitative triplets into one folder you can download.

In [ ]:
import shutil, os
os.makedirs('submission_assets', exist_ok=True)
for out in ['outputs_full','outputs_l1']:
    if os.path.exists(f'{out}/loss_curve.png'):
        shutil.copy(f'{out}/loss_curve.png', f'submission_assets/loss_curve_{out}.png')
    if os.path.exists(f'{out}/loss_log.csv'):
        shutil.copy(f'{out}/loss_log.csv', f'submission_assets/loss_log_{out}.csv')
    sdir = f'{out}/samples'
    if os.path.isdir(sdir):
        dst = f'submission_assets/samples_{out}'; os.makedirs(dst, exist_ok=True)
        for f in sorted(os.listdir(sdir))[-8:]:
            shutil.copy(os.path.join(sdir,f), os.path.join(dst,f))
print('Assets ready in submission_assets/:')
for f in sorted(os.listdir('submission_assets')): print(' -', f)
print('\nWeights to upload publicly:')
print(' - outputs_full/final_weights.pth  (this is the model you submit)')


## 10. Next steps after this finishes
1. Check the smoke-test ran clean, then set `SMOKE = False` (cell 1) and **Run All** again for the real run.
2. **Download** `outputs_full/final_weights.pth` and `submission_assets/` (Output tab → download).
3. **Upload weights** to Hugging Face Hub (public) → put the link in README + the submission form.
4. **Fill in** the `[fill after training]` cells in `Technical_Report.pdf`/`REPORT.md` and the README results table using the printed metrics, loss curves, and 5 triplets.
5. Commit everything to your public GitHub repo and build `Krishna_Kant_GalaxEye.zip` (report PDF + time log + loss curves + triplet images; **not** the weights).
